Just some random musings on various snippets of BPL I want to understand more about.

## Spacing, Sign and Multiplication

In [ ]:
%%bpl
⍝ 4 followed by monadic +3, which gives 3. Result: vector 4 3.
4 +3

4 3

In [ ]:
%%bpl
⍝ 4 followed by the sign of 3, which is 1. Result: vector 4 1.
4 ×3

4 1ₓ

In [ ]:
%%bpl
⍝ One run: multiply 4 by 3. Result: 12.
4×3

12

In [ ]:
%%bpl
⍝ Multiply the two inputs. Result: 12.
4 × 3

12

In [ ]:
%%bpl
⍝ Evaluate the joined run 4+1 first: (4+1) × 3. Result: 15.
4+1 × 3

15

In [ ]:
%%bpl
⍝ Evaluate right to left: 4 + (1 × 3). Result: 7.
4 + 1 × 3

7

In [ ]:
%%bpl
⍝ Evaluate 4+1 and the sign of 3 separately. Result: vector 5 1.
4+1 ×3

5 1ₓ

`×` is a function: with two inputs it multiplies; with one input it returns the sign. For real numbers, the sign is `1` for positive values, `0` for zero and `¯1` for negative values.

In BPL, spaces separate groups called runs. Each run is evaluated before the surrounding expression. A run starting with a function, such as `×3`, is a monadic call. Separate values next to each other form a vector.

`4 + 1 × 3` follows right-to-left evaluation, giving `7`. This is not a rule that multiplication has higher precedence than addition. Spaces can change grouping in BPL; parentheses make the intended grouping explicit.

[BPL evaluation guide](https://answerdotai.github.io/basedpl/evaluation.html).

## The Alphabet in APL and BPL: Taking Letters with ↑

APL uses `⎕A` (Quad A) for the uppercase Latin alphabet. BPL uses **`$a`** instead; it is a built-in character vector. The `⎕A` example above fails in BPL because it encounters the undefined name `A`.

The up arrow `↑` is a function. With two inputs it means **take**: a positive count takes letters from the start, and a negative count takes them from the end. With one input it means **first**. These examples use BPL's `$a` alphabet constant.

[BPL alphabet reference](https://answerdotai.github.io/basedpl/system/constants.html#a-alphabet).

In [ ]:
%%bpl
⍝ BPL alphabet constant (APL uses ⎕A)
$a

⍝ Dyadic ↑: take the first three letters. Result: ABC.
3 ↑ $a

⍝ A negative count takes from the end. Result: XYZ.
¯3 ↑ $a

⍝ Monadic ↑: first letter. Result: A.
↑ $a

┌→─────────────────────────┐
│ABCDEFGHIJKLMNOPQRSTUVWXYZ│
└──────────────────────────┘

┌→──┐
│ABC│
└───┘

┌→──┐
│XYZ│
└───┘

'A'

## Means with the Under Operator (@)

This example studies four kinds of mean using the same averaging function.

`@` is the **Under operator**. For the invertible numeric functions used here, `F@G` creates a new function that does three things: **apply G, apply F, then apply the inverse of G**. In notation: `(F@G) y` means `G⁻¹(F(G y))`. The result of G goes into F, and then the inverse transform converts the answer back.

`+/÷≢` is an arithmetic-mean function train: `+/` sums the input, `≢` counts its items, and `÷` divides the sum by the count. For `3 5 7`, that is `15÷3`, giving `5`.

In [ ]:
%%bpl
⍝ Define the average function
avg ← +/÷≢

In [ ]:
%%bpl
⍝ Arithmetic mean: sum divided by count
avg 3 5 7

5

**Geometric mean:** first meet `⍟` (circle star). With one input, this function takes the **natural logarithm** of each number. Its inverse is monadic `*` (star), the **exponential** function: exponential turns those logarithms back into the original numbers. Here `*` has one input, so it means exponential, not two-input power.

In `(avg@⍟) 3 5 7`, `@` creates a function that takes the logs, averages them, then applies exponential to that average. Read it as "average under logarithm". You do not need to calculate the inverse yourself: `@` supplies it. For these positive inputs, the result is the geometric mean, about **4.717694**. The next cell shows each stage before the compact version.

In [ ]:
%%bpl
⍝ 1. Natural logarithm of each input
⍟ 3 5 7

⍝ 2. Average those logarithms
avg (⍟ 3 5 7)

⍝ 3. Exponential of that average: geometric mean
* (avg (⍟ 3 5 7))

⍝ Under performs all three steps
(avg@⍟) 3 5 7

1.098612289 1.609437912 1.945910149

1.551320117

4.71769398

4.71769398

The cell above contains **four separate expressions**. The first three let us inspect increasingly complete calculations. The last expression, `(avg@⍟) 3 5 7`, repeats the whole calculation using Under. It does not reuse the earlier outputs: each expression calculates afresh.

Here is exactly how that last line is evaluated:

1. **Create the function: `(avg@⍟)`.** The operator `@` takes two functions as its operands: `avg` on its left and `⍟` on its right. At this point we are combining functions, not averaging any numbers yet.
2. **Use Under's built-in recipe.** For this logarithm transform, the function created by `@` behaves as `* (avg (⍟ input))`: apply the right operand, apply the left operand, then undo the right operand. BPL knows that monadic `*` (exponential) is the inverse of monadic `⍟` (natural logarithm). This is why an exponential happens even though there is no `*` in the compact expression. The automatic inverse stage is part of what `@` means.
3. **Call the new function with `3 5 7`.** This is one argument: a vector containing three numbers. The parentheses group the derived function, and the vector on its right is its input.
4. **Apply `⍟` to the input vector.** Each number becomes its natural logarithm, giving approximately `1.098612289 1.609437912 1.945910149`. This vector is passed to `avg`.
5. **Run `avg` on those three logarithms.** Its fork sums them to about `4.653960350` and counts three items, then divides the sum by three. The result is one number: approximately `1.551320117`. It is averaging the logarithms, not the original `3 5 7`.
6. **Apply the inverse transform to that one result.** Monadic `*` calculates the exponential of `1.551320117`, giving approximately `4.71769398`. Under applies this inverse to the average, not separately to the original inputs.
7. **Return `4.71769398`.** That is the geometric mean.

The two complete expressions below therefore do the same calculation:

```apl
* (avg (⍟ 3 5 7))
(avg@⍟) 3 5 7
```

`@` does not change the definition of `avg` or `⍟`. It creates a new function that uses them in this specified order, with the inverse of `⍟` as its final stage. [BPL Under reference](https://answerdotai.github.io/basedpl/glyphs/at.html#other-functions).

**Harmonic mean:** `÷` is the division glyph, also called obelus. With two inputs, `6 ÷ 3` divides and gives `2`. With one input, `÷ 3` means **reciprocal**: `1÷3`. On a vector, it takes the reciprocal of each number.

Reciprocal undoes itself: the reciprocal of `1÷3` is `3`. So `(avg@÷) 3 5 7` takes `1÷3`, `1÷5` and `1÷7`, averages them to `71÷315`, then takes the reciprocal of that average: **315÷71**, about **4.436620**. That is the harmonic mean. Here the transform `÷` is monadic; it is not dividing two separate arguments.

In [ ]:
%%bpl
⍝ 1. Reciprocal of each input: 1/3, 1/5, 1/7
÷ 3 5 7

⍝ 2. Average those reciprocals
avg (÷ 3 5 7)

⍝ 3. Reciprocal of that average: harmonic mean
÷ (avg (÷ 3 5 7))

⍝ Under performs all three steps
(avg@÷) 3 5 7

0.3333333333 0.2 0.1428571429

0.2253968254

4.436619718

4.436619718

**Root mean square (RMS):** `⊼` is the **wedge bar** glyph, a single character. In BPL, its one-input meaning is **square**: `⊼ 3` gives `9`. On `3 5 7`, it gives `9 25 49`. The inverse used here is `√` (radical), the **square-root** function: `√ 9` gives `3`.

So `(avg@⊼) 3 5 7` squares each input, averages the squares to `83÷3`, then takes the square root, giving about **5.259911**. Read the name backwards through the calculation: square → mean → root. `@` supplies the square root automatically; the expanded version below writes it explicitly. These examples use positive inputs.

In [ ]:
%%bpl
⍝ 1. Square each input
⊼ 3 5 7

⍝ 2. Average the squares
avg (⊼ 3 5 7)

⍝ 3. Square root of that average: root mean square
√ (avg (⊼ 3 5 7))

⍝ Under performs all three steps
(avg@⊼) 3 5 7

9 25 49

27.66666667

5.259911279

5.259911279

All four examples use the same `avg` function; Under changes the space in which we average.

| Mean | Derived function | Steps on the input | Result for 3 5 7 |
|---|---|---|---|
| Arithmetic | `avg` | Average directly | `5` |
| Geometric | `avg@⍟` | Natural log → average → exponential | About `4.717694` |
| Harmonic | `avg@÷` | Reciprocal → average → reciprocal | `315/71`, about `4.436620` |
| Root mean square | `avg@⊼` | Square → average → square root | About `5.259911` |

[Under reference](https://answerdotai.github.io/basedpl/glyphs/at.html#other-functions). The Python example below organises these same functions into a named group.

In [ ]:
# Python cell: organise the BPL functions and apply each to a
from basedpl import Array, fn

a = Array([3, 5, 7])
mean = "+/÷≢"
means = dict(
    arithmetic=mean,
    geometric=f"({mean})@⍟",
    harmonic=f"({mean})@÷",
    root_mean_square=f"({mean})@⊼",
)

mean_funcs = {name: fn(source) for name, source in means.items()}
results = Array({name: func(a) for name, func in mean_funcs.items()})
results


["arithmetic":5ₓ "geometric":4.71769398 "harmonic":315ᵣ71 "root_mean_square":5.259911279]

Read the Python example in stages:

1. `Array([3, 5, 7])` makes a BPL array from a Python list. This is the common input for every mean.
2. `mean = "+/÷≢"` stores BPL source as a Python string. It has not called the averaging function yet.
3. The `means` dictionary gives a name to each source string. For example, `f"({mean})@⍟"` inserts the text of `mean`, producing `(+/÷≢)@⍟`. These braces are **Python f-string interpolation**, not a BPL function body. The parentheses group the averaging train as the left operand of `@`.
4. `fn(source)` turns each BPL definition into a callable function. The dictionary comprehension stores those functions under the same names.
5. `func(a)` calls each function with the same array. Wrapping the named results in `Array(...)` displays them as a keyed BPL array.

The screenshot collects the functions into a keyed `Array` and calls the whole collection with `mean_funcs(a)`. Your installed BPL **0.1.55** does not support that final grouped call (`Array` is not callable), so the runnable version above uses a Python dictionary and explicitly calls each function. The four Under expressions and their results are the same.

`@` is a **dyadic operator** because it takes two operands: the averaging function and the transform. The function it creates is called **monadically** here, with one argument: the vector `3 5 7`.